In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import shutil
import random
from pathlib import Path

# ==========================================
# 1. 환경 설정
# ==========================================
SOURCE_DIR = "/content/drive/MyDrive/AI_HUB_RAW_DATA"     # 원본 데이터 최상위 폴더
TARGET_DIR = "/content/drive/MyDrive/Recycle_Dataset"     # 새롭게 정리될 폴더
SAMPLE_COUNT = 1500                  # 클래스당 추출할 목표 이미지 수
TRAIN_RATIO = 0.8                    # 학습용(Train) 비율

TARGET_CLASSES = {
    "Paper": "종이류",
    "Can": "캔류",
    "Glass": "유리병",
    "PET": "페트병",
    "Vinyl": "비닐"
}

# ==========================================
# 2. 폴더 생성 함수
# ==========================================
def create_dir(path):
    if not os.path.exists(path):
        os.makedirs(path)

train_dir = os.path.join(TARGET_DIR, "train")
val_dir = os.path.join(TARGET_DIR, "val")
create_dir(train_dir)
create_dir(val_dir)

print("🚀 중첩 폴더 탐색 및 데이터 샘플링을 시작합니다...\n")

for class_name, raw_folder_name in TARGET_CLASSES.items():
    raw_path = Path(SOURCE_DIR) / raw_folder_name

    if not raw_path.exists():
        print(f"⚠️ 경고: 원본 경로를 찾을 수 없습니다. ({raw_path})")
        continue

    # ==========================================
    # 하위 폴더를 모두 뒤져서 이미지 파일 경로 수집
    # ==========================================
    print(f"🔍 [{class_name}] 하위 폴더의 모든 이미지를 스캔 중입니다...")
    all_images = []
    # 다양한 확장자 지원을 위해 리스트로 묶어서 탐색
    for ext in ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.PNG']:
        all_images.extend(list(raw_path.rglob(ext)))

    if not all_images:
        print(f"⚠️ '{class_name}' 폴더 내에 이미지가 없습니다.")
        continue

    actual_sample_count = min(SAMPLE_COUNT, len(all_images))
    if len(all_images) < SAMPLE_COUNT:
        print(f"   ㄴ 목표치({SAMPLE_COUNT}장)보다 적습니다. (총 {len(all_images)}장 발견)")

    # 무작위 샘플링
    sampled_images = random.sample(all_images, actual_sample_count)

    # Train / Val 분할
    train_count = int(actual_sample_count * TRAIN_RATIO)
    train_images = sampled_images[:train_count]
    val_images = sampled_images[train_count:]

    # 타겟 폴더 생성
    class_train_dir = Path(train_dir) / class_name
    class_val_dir = Path(val_dir) / class_name
    create_dir(class_train_dir)
    create_dir(class_val_dir)

    # ==========================================
    # 파일 복사 시 이름 중복 방지를 위한 이름 재지정
    # ==========================================
    # Train 데이터 복사
    for idx, img_path in enumerate(train_images):
        new_filename = f"{class_name}_train_{idx:04d}{img_path.suffix}" # 예: Vinyl_train_0000.jpg
        shutil.copy(img_path, class_train_dir / new_filename)

    # Validation 데이터 복사
    for idx, img_path in enumerate(val_images):
        new_filename = f"{class_name}_val_{idx:04d}{img_path.suffix}"   # 예: Vinyl_val_0000.jpg
        shutil.copy(img_path, class_val_dir / new_filename)

    print(f"✅ [{class_name}] 추출 완료: Train {len(train_images)}장, Val {len(val_images)}장\n")

print("🎉 모든 데이터 샘플링 및 폴더 정리가 완벽하게 끝났습니다!")

In [ ]:
# 1. 구글 드라이브에 있는 압축 파일을 코랩의 고속 로컬 디렉토리(/content/)로 복사
!cp /content/drive/MyDrive/Recycle_Dataset.zip /content/

# 2. 코랩 내부에서 압축 풀기 (출력 메시지를 숨기기 위해 -q 옵션 사용)
!unzip -q /content/Recycle_Dataset.zip -d /content/recycle_local

replace /content/recycle_local/Recycle_Dataset/train/Can/Can_train_0000.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/recycle_local/Recycle_Dataset/train/Can/Can_train_0001.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/recycle_local/Recycle_Dataset/train/Can/Can_train_0002.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: A


In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

# ==========================================
# 하드웨어 가속기(GPU) 설정 및 배치 사이즈
# ==========================================
BATCH_SIZE = 32

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ 현재 사용 중인 디바이스: {device} (T4 GPU가 정상 작동 중인지 확인하세요!)")

# ==========================================
# 3. 데이터 증강 및 전처리 설정
# ==========================================
class_names = ['Can', 'Glass', 'PET', 'Paper', 'Vinyl']

data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),   # 좌우 반전 데이터 증강
        transforms.RandomRotation(15),       # 회전 데이터 증강
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

# ==========================================
# 구글 드라이브 내 데이터셋 경로 설정
# ==========================================
TARGET_DIR = "/content/drive/MyDrive/Recycle_Dataset"

try:
    image_datasets = {
        'train': datasets.ImageFolder(f"{TARGET_DIR}/train", data_transforms['train']),
        'val': datasets.ImageFolder(f"{TARGET_DIR}/val", data_transforms['val'])
    }

    dataloaders = {
        'train': DataLoader(image_datasets['train'], batch_size=BATCH_SIZE, shuffle=True, num_workers=2),
        'val': DataLoader(image_datasets['val'], batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    }

    class_names = image_datasets['train'].classes
    print(f"✅ 학습할 클래스 목록: {class_names}")
    print(f"📊 학습 데이터 개수: {len(image_datasets['train'])}개 / 검증 데이터 개수: {len(image_datasets['val'])}개")

except FileNotFoundError:
    print(f"❌ 에러: {TARGET_DIR} 경로에서 폴더를 찾을 수 없습니다.")
    print("💡 팁: 구글 드라이브에 업로드한 폴더명이 정확히 'Recycle_Dataset'인지, 혹은 다른 하위 폴더에 들어있지 않은지 확인해 주세요!")

# ==========================================
# 전이 학습 모델(ResNet-50) 및 출력층 설정
# ==========================================
print("🔄 ResNet-50 사전 학습 모델을 다운로드하는 중입니다...")
model = models.resnet50(pretrained=True)

# 최종 출력 레이어를 우리 데이터셋의 클래스 개수에 맞게 수정
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, len(class_names))

# 모델을 GPU(Cuda) 메모리로 이동
model = model.to(device)

print("🎉 [코랩 환경] 데이터 로더 구축 및 ResNet-50 모델 셋업 완벽 완료!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ 현재 사용 중인 디바이스: cuda (T4 GPU가 정상 작동 중인지 확인하세요!)
✅ 학습할 클래스 목록: ['Can', 'Glass', 'PET', 'Paper', 'Vinyl']
📊 학습 데이터 개수: 6000개 / 검증 데이터 개수: 1500개
🔄 ResNet-50 사전 학습 모델을 다운로드하는 중입니다...
🎉 [코랩 환경] 데이터 로더 구축 및 ResNet-50 모델 셋업 완벽 완료!


In [ ]:
import time
import copy

# ==========================================
# 1. 손실 함수(Loss) 및 최적화 함수(Optimizer) 설정
# ==========================================
# 다중 클래스 분류이므로 CrossEntropyLoss를 사용합니다.
criterion = nn.CrossEntropyLoss()

# 사전 학습된 가중치를 미세 조정(Fine-tuning)하기 위해 Adam 옵티마이저를 사용합니다.
# 학습률(Learning Rate)은 전이 학습에서 안전한 0.0001(1e-4)로 설정했습니다.
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

# ==========================================
# 2. 모델 학습 및 검증 통합 함수 정의
# ==========================================
def train_model(model, criterion, optimizer, num_epochs=10):
    since = time.time()

    # 가장 좋은 성능을 낸 모델의 가중치를 저장하기 위한 변수
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    for epoch in range(num_epochs):
        print(f'\n▶ Epoch {epoch + 1}/{num_epochs}')
        print('-' * 20)

        # 각 에포크는 학습(train)과 검증(val) 단계를 가집니다.
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()  # 모델을 학습 모드로 설정
            else:
                model.eval()   # 모델을 평가 모드로 설정

            running_loss = 0.0
            running_corrects = 0

            # 데이터 로더로부터 배치 단위로 데이터를 가져옴
            for inputs, labels in dataloaders[phase]:
                # 데이터를 GPU(device)로 이동
                inputs = inputs.to(device)
                labels = labels.to(device)

                # 옵티마이저 그라디언트 초기화
                optimizer.zero_grad()

                # 순전파(Forward) 연산
                # 학습 단계에서만 연산 기록을 추적(배치 정규화, 드롭아웃 등 활성화)
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    # 학습 단계에서만 역전파(Backward) 및 가중치 업데이트 수행
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                # 통계치 계산 (배치별 손실과 정답 개수 누적)
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            # 에포크 전체의 평균 손실(Loss)과 정확도(Accuracy) 계산
            epoch_loss = running_loss / len(image_datasets[phase])
            epoch_acc = running_corrects.double() / len(image_datasets[phase])

            print(f'[{phase.upper()}] Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            # 검증(val) 단계에서 성능이 기존 최고 기록보다 좋으면 가중치를 업데이트
            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())

    # 전체 학습 시간 출력
    time_elapsed = time.time() - since
    print(f'\n🎉 학습 완료! 소요 시간: {time_elapsed // 60:.0f}분 {time_elapsed % 60:.0f}초')
    print(f'🏆 최적의 검증 정확도(Best Val Acc): {best_acc:.4f}')

    # 가장 우수한 성능을 보였던 모델 가중치를 불러와 반환
    model.load_state_dict(best_model_wts)
    return model

# ==========================================
# 3. 모델 학습 시작
# ==========================================
# 기본 에포크(반복 횟수)를 10회로 설정하여 학습을 진행합니다.
# 데이터 개수나 성능에 따라 num_epochs를 조정해 보세요.
NUM_EPOCHS = 10

print("🚀 모델 학습 및 검증을 시작합니다...")
model = train_model(model, criterion, optimizer, num_epochs=NUM_EPOCHS)

🚀 모델 학습 및 검증을 시작합니다...

▶ Epoch 1/10
--------------------
[TRAIN] Loss: 0.5310 Acc: 0.8068
[VAL] Loss: 0.3175 Acc: 0.8773

▶ Epoch 2/10
--------------------
[TRAIN] Loss: 0.2781 Acc: 0.8998
[VAL] Loss: 0.2919 Acc: 0.8953

▶ Epoch 3/10
--------------------
[TRAIN] Loss: 0.2088 Acc: 0.9257
[VAL] Loss: 0.2953 Acc: 0.8973

▶ Epoch 4/10
--------------------
[TRAIN] Loss: 0.1612 Acc: 0.9437
[VAL] Loss: 0.3206 Acc: 0.8940

▶ Epoch 5/10
--------------------
[TRAIN] Loss: 0.1332 Acc: 0.9557
[VAL] Loss: 0.3535 Acc: 0.8907

▶ Epoch 6/10
--------------------
[TRAIN] Loss: 0.1130 Acc: 0.9602
[VAL] Loss: 0.2819 Acc: 0.9153

▶ Epoch 7/10
--------------------
[TRAIN] Loss: 0.0786 Acc: 0.9737
[VAL] Loss: 0.3162 Acc: 0.9093

▶ Epoch 8/10
--------------------
[TRAIN] Loss: 0.0820 Acc: 0.9715
[VAL] Loss: 0.2936 Acc: 0.9107

▶ Epoch 9/10
--------------------
[TRAIN] Loss: 0.0856 Acc: 0.9712
[VAL] Loss: 0.3609 Acc: 0.9080

▶ Epoch 10/10
--------------------
[TRAIN] Loss: 0.0722 Acc: 0.9775
[VAL] Loss: 0.351

In [ ]:
import torch

# 저장할 파일 이름 정의
MODEL_SAVE_PATH = '/content/recycle_best_model.pth'

# 3단계에서 학습이 완료된 'model' 객체의 가중치(state_dict)를 저장합니다.
torch.save(model.state_dict(), MODEL_SAVE_PATH)

print(f"✅ 모델 가중치가 [{MODEL_SAVE_PATH}] 경로에 성공적으로 저장되었습니다!")

✅ 모델 가중치가 [/content/recycle_best_model.pth] 경로에 성공적으로 저장되었습니다!


In [ ]:
!cp /content/recycle_best_model.pth /content/drive/MyDrive/

print("✅ 구글 드라이브(내 드라이브)로 모델 백업이 완료되었습니다!")

cp: cannot stat '/content/recycle_best_model.pth': No such file or directory
✅ 구글 드라이브(내 드라이브)로 모델 백업이 완료되었습니다!


In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision import models
from PIL import Image
import matplotlib.pyplot as plt
from google.colab import files
import io
import os

# ==========================================
# 0. 5개 클래스 기반 모델 빌드 및 가중치 강제 로드
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_names = ['Can', 'Glass', 'PET', 'Paper', 'Vinyl']

print("🔄 구글 드라이브에서 학습 완료된 가중치를 새로 불러옵니다...")
standalone_model = models.resnet50(pretrained=False)
num_features = standalone_model.fc.in_features
standalone_model.fc = nn.Linear(num_features, len(class_names)) # 딱 5개로 고정

# 구글 드라이브 경로 지정
LOAD_PATH = '/content/drive/MyDrive/recycle_best_model.pth'

# 🚨 파일 존재 여부 철저히 체크 (압축 안 풀렸으면 여기서 경고가 뜹니다)
if not os.path.exists(LOAD_PATH):
    print("\n❌ [비상 에러] 구글 드라이브에서 'recycle_best_model.pth' 파일을 찾을 수 없습니다!")
    print("💡 해결책: 구글 드라이브 내드라이브 최상위에 압축을 푼 'recycle_best_model.pth' 파일이 잘 들어있는지 꼭 확인하세요.")
else:
    # 파일이 존재할 때만 가중치 탑재
    standalone_model.load_state_dict(torch.load(LOAD_PATH, map_location=device))
    print("🎯 [성공] 가중치 파일을 불러왔습니다.")

    standalone_model = standalone_model.to(device)

    # ==========================================
    # 1. 즉석 파일 업로드 및 추론
    # ==========================================
    print("\n📂 분류하고 싶은 쓰레기 사진을 업로드해 주세요.")
    uploaded = files.upload()

    if not uploaded or len(uploaded) == 0:
        print("⚠️ 선택된 사진이 없습니다.")
    else:
        file_name = list(uploaded.keys())[0]

        image = Image.open(io.BytesIO(uploaded[file_name])).convert('RGB')
        infer_transforms = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])

        input_tensor = infer_transforms(image).unsqueeze(0).to(device)

        standalone_model.eval()
        with torch.no_grad():
            outputs = standalone_model(input_tensor)
            probabilities = torch.nn.functional.softmax(outputs[0], dim=0)
            prob, preds = torch.max(probabilities, 0)

            pred_index = preds.item()
            confidence_score = prob.item() * 100
            predicted_class = class_names[pred_index]

        # 결과 시각화
        plt.figure(figsize=(6, 6))
        plt.imshow(image)
        plt.axis('off')

        result_title = f"Prediction: {predicted_class} ({confidence_score:.2f}%)"
        title_color = 'green' if confidence_score > 70 else 'orange'
        plt.title(result_title, fontsize=16, color=title_color, weight='bold')
        plt.show()

        print(f"🎯 [분석 결과]: 해당 쓰레기는 {confidence_score:.2f}%의 확률로 [{predicted_class}] 카테고리입니다.")

In [ ]:
%%writefile app.py

import streamlit as st
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision import models
from PIL import Image
import io

# ==========================================
# 1. 페이지 기본 설정 및 디자인
# ==========================================
st.set_page_config(page_title="K-분리배출 도우미 AI", page_icon="♻️", layout="centered")

st.title("♻️ AI 기반 실시간 분리수거 도우미")
st.write("쓰레기 사진을 업로드하거나 촬영하면 인공지능이 올바른 분리배출 카테고리를 알려줍니다.")
st.markdown("---")

# 환경 및 클래스 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_names = ['Can', 'Glass', 'PET', 'Paper', 'Vinyl']

# 분리배출 꿀팁 메시지 매핑 (발표할 때 UX 개선 점수를 받기 좋습니다!)
recycle_tips = {
    'Can': "🥫 **캔류 고도화 배출 팁:** 내용물을 완전히 비우고 물로 헹군 뒤, 가능한 압착해서 캔류 수거함에 버려주세요!",
    'Glass': "🍾 **유리병 배출 팁:** 담배꽁초 등 이물질을 넣지 말고, 플라스틱이나 알루미늄 뚜껑은 완전히 제거한 뒤 배출하세요.",
    'PET': "🥤 **페트병 배출 팁:** 라벨(비닐)을 전면 제거하고 내부를 세척한 뒤, 꾹 압착하여 투명 페트병 전용 수거함에 넣어주세요!",
    'Paper': "📦 **종이류 배출 팁:** 택배 박스의 테이프, 송장 스티커, 철핀 등을 완전히 제거하고 접어서 배출해 주세요.",
    'Vinyl': "🛍️ **비닐류 배출 팁:** 과자봉지, 리필용기 등 비닐류는 이물질을 깨끗이 씻어 투명 비닐봉투에 모아서 배출하세요."
}

# ==========================================
# 2. 캐싱 기능을 활용한 딥러닝 모델 로드
# ==========================================
@st.cache_resource
def load_my_model():
    # 뼈대 생성
    model = models.resnet50(pretrained=False)
    num_features = model.fc.in_features
    model.fc = nn.Linear(num_features, len(class_names))

    # 구글 드라이브에 연결된 최적 가중치 파일 로드
    LOAD_PATH = '/content/drive/MyDrive/recycle_best_model.pth'
    model.load_state_dict(torch.load(LOAD_PATH, map_location=device))
    model = model.to(device)
    model.eval()
    return model

try:
    model = load_my_model()
    st.success("🎯 정상 작동 중입니다!")
except Exception as e:
    st.error("❌ 구글 드라이브에서 'recycle_best_model.pth' 파일을 로드하지 못했습니다.")
    st.info("💡 팁: 드라이브 최상단에 압축을 푼 pth 파일이 정상적으로 존재하는지 확인하세요.")

# ==========================================
# 3. 사용자 사진 입력 (파일 업로드 및 카메라 촬영 동시에 지원)
# ==========================================
uploaded_file = st.file_uploader("분류할 쓰레기 사진을 올려주세요...", type=["jpg", "jpeg", "png"])

if uploaded_file is not None:
    # 이미지 열기
    image = Image.open(uploaded_file).convert('RGB')

    # 두 개의 칸으로 나누어 UI 배치 (왼쪽: 이미지, 오른쪽: AI 분석 결과)
    col1, col2 = st.columns(2)

    with col1:
        st.image(image, caption="업로드된 사진", use_container_width=True)

    with col2:
        st.subheader("🤖 인공지능 분석 결과")

        # 이미지 전처리 (코랩 실험 데이터와 100% 동일한 트랜스폼 적용)
        infer_transforms = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])

        input_tensor = infer_transforms(image).unsqueeze(0).to(device)

        # 추론 수행
        with torch.no_grad():
            outputs = model(input_tensor)
            probabilities = torch.nn.functional.softmax(outputs[0], dim=0)
            prob, preds = torch.max(probabilities, 0)

            predicted_class = class_names[preds.item()]
            confidence_score = prob.item() * 100

        # 결과 화면 표시
        st.metric(label="예측된 카테고리", value=predicted_class)
        st.progress(int(confidence_score))
        st.write(f"📊 **판단 확신도:** {confidence_score:.2f}%")

        st.markdown("---")
        # 해당 품목에 맞는 분리배출 꿀팁 안내
        st.info(recycle_tips[predicted_class])

Writing app.py


In [ ]:
# 1. 기존에 혹시 열려있을지 모르는 ngrok 터널 청소 (중복 실행 에러 방지)
try:
    from pyngrok import ngrok
    ngrok.kill()
except:
    pass

# 2. 필수 패키지 설치 (-q 옵션을 주어 설치 과정을 화면에 띄우지 않고 조용히 설치합니다)
print("📦 웹 엔진 패키지 설치 중...")
!pip install -q streamlit pyngrok

# 3. 본인의 인증 토큰 등록 (깃허브 제출을 위해 개인 토큰은 숨김 처리했습니다)
!ngrok config add-authtoken "YOUR_NGROK_AUTHTOKEN"

# 4. 백그라운드에서 스트림릿 서버 실행
print("🚀 웹 서버(app.py)를 가동합니다...")
import subprocess
subprocess.Popen(["streamlit", "run", "app.py"])

# 5. ngrok 터널링 가동 (8501 포트 열기)
print("🌐 터널 링크를 생성하는 중...")
from pyngrok import ngrok
public_url = ngrok.connect(8501)

print("\n" + "="*50)
print(f"🔗 아래 주소를 클릭하거나 스마트폰으로 접속하세요:")
print(f"{public_url}")
print("="*50)

📦 웹 엔진 패키지 설치 중...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 116.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 114.8 MB/s eta 0:00:00
Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml
🚀 웹 서버(app.py)를 가동합니다...
🌐 터널 링크를 생성하는 중...

🔗 아래 주소를 클릭하거나 스마트폰으로 접속하세요:
NgrokTunnel: "https://resurrect-whoops-richness.ngrok-free.dev" -> "http://localhost:8501"
